# Multithreading

## 1. Processes vs Threads
Think of a computer as building

**Process** = a separate apartment.

| Apartment has | Process has |
| :-- | :-- |
| Its own rooms | Its own memory |
| Its own furniture | Its own resources |
| Its own electricity | Cannot directly access another process' memory |

<br>

**Thread** = individual in the apartment

Threads belong to a process. They all:
- share memory
- share variables
- run concurrently

This sharing of resources makes them fast but introduces problems such as race conditions.

| Process | Thread |
| :-- | :-- |
| Heavy | Lightweight |
| Own memory | Shared memory |
| Slower creation | Faster creation |
| Safer | Less safe |
| Inter-process communication needed | Access same variables |

## 2. Semaphore
E.g. Car Park
```
Parking Capacity = 2 cars

Cars arrive:        Only 2 cars can enter:
A B C D             Parked:
                    A B
                    Waiting:
                    C D


Car A leaves        Car B leaves
Parked:             Parked:
B C                 C D
Waiting:
D
```

Semaphore maintains the limit and queue

In [1]:
import threading
import time

def current_time():
    return time.strftime("%H:%M:%S")


In [ ]:
semaphore = threading.Semaphore(2)
def worker(name):
    with semaphore:
        print(f"{name} entered at {current_time()}")
        time.sleep(2)
        print(f"{name} exited at {current_time()}")

for n in ["A", "B", "C", "D"]:
    threading.Thread(target=worker, args=(n,)).start()

A entered at 15:07:47
B entered at 15:07:47


A exited at 15:07:49
C entered at 15:07:49
B exited at 15:07:49
D entered at 15:07:49
C exited at 15:07:51
D exited at 15:07:51


Only 2 workers run simultaneously.

## 3. Creating Threads

In [3]:
import threading
def hello():
    print("Hello")
t= threading.Thread(target=hello)
t.start()                           # 1. Starts the thread
t.join()                            # 2. Waits for the thread to finish

Hello


Without `t.join()` the main program may finish before thread.

With `t.join()` it waits for the thread to finish before continuing.

## 4. Multiple Threads

In [5]:
import threading
import time
def task(name):
    print(f"{name} started @ {current_time()}")
    time.sleep(2)
    print(f"{name} finished @ {current_time()}")

t1 = threading.Thread(target=task, args=("E",))
t2 = threading.Thread(target=task, args=("F",))
t1.start()
t2.start()
t1.join()
t2.join()


E started @ 15:08:07
F started @ 15:08:07
E finished @ 15:08:09
F finished @ 15:08:09


## 5. Thread IDs
Different threads have different IDs - used for debugging. 

In [6]:
def worker():
    print(threading.get_ident())
threading.Thread(target=worker).start()

15304


## 6. *args
Unknown number of positional arguments

In [7]:
def arguments(*args):
    print(args)
arguments(1,2,3)
arguments()

(1, 2, 3)
()


The output is a tuple - can loop or index, but is immutable.

In [8]:
def add(*args):
    print(sum(args))
add(1,2,3)
add(4,5,6)

6
15


## 7. *kwargs
Unknown number of named arguments

In [9]:
def user(**kwargs):
    print(kwargs)
user(name='Jonathan', age=30)

{'name': 'Jonathan', 'age': 30}


kwargs becomes a dictionary

In [10]:
def user(**kwargs):
    for key, value in kwargs.items():
        print(key, value)
user(
     name   = "John"
    ,age    = 25
    ,city   = "London"
)

name John
age 25
city London


## 8. Using *args and **kwargs together
Order matters:

In [11]:
def example(*args, **kwargs):
    print(args)
    print(kwargs)

example(
    1,2,3
    ,name   = "Jane"
    ,age    = 25
    ,job    = "Python developer"
)

(1, 2, 3)
{'name': 'Jane', 'age': 25, 'job': 'Python developer'}


## 9. Race Conditions
If two threads modify the same variable:

In [13]:
counter = 0
def increase(by):
    global counter
    local_counter = counter
    local_counter += by
    time.sleep(0.1)
    counter = local_counter
    print(f'counter={counter}')
t1 = threading.Thread(target=increase, args=(10,))
t2 = threading.Thread(target=increase, args=(20,))
t1.start()
t2.start()
t1.join()
t2.join()

print(f'The final counter is {counter}')


counter=10
counter=20
The final counter is 20


Both threads read the same value before either writes it back.

## 10. Lock
A lock protects critical code

In [14]:
counter = 0

def increase(by, lock):
    global counter
    lock.acquire()
    local_counter = counter
    local_counter += by
    time.sleep(0.1)
    counter = local_counter
    print(f'counter={counter}')
    lock.release()
lock = threading.Lock()
t1 = threading.Thread(target=increase, args=(10, lock))
t2 = threading.Thread(target=increase, args=(20, lock))
t1.start()
t2.start()
t1.join()
t2.join()
print(f'The final counter is {counter}')

counter=10
counter=30
The final counter is 30


Only one thread can enter at a time, this prevents race conditions.

## 11. Deadlock
> Thread 1 is waiting for lock 2<br>
Thread 2 is waiting for lock 1

In [15]:
lock1 = threading.Lock()
lock2 = threading.Lock()

def thread_a():
    with lock1:
        print("A: acquired lock1, waiting for lock2")
        time.sleep(0.1)
        with lock2:
            print("A: acquired both locks")

def thread_b():
    with lock2:
        print("B: acquired lock2, waiting for lock1")
        time.sleep(0.1)
        with lock1:
            print("B: acquired both locks")
t1 = threading.Thread(target=thread_a)
t2 = threading.Thread(target=thread_b)
t1.start()
t2.start()
t1.join(timeout=2)
t2.join(timeout=2)
print("Thread A alive:", t1.is_alive())
print("Thread B alive:", t2.is_alive())


A: acquired lock1, waiting for lock2
B: acquired lock2, waiting for lock1
Thread A alive: True
Thread B alive: True


## 12. Reentrant Lock (RLock)


In [16]:
shared_lock = threading.RLock() # allows the same thread to acquire the lock multiple times without deadlock.
lock1 = shared_lock
lock2 = shared_lock
def thread_a():
    lock1.acquire() # threading.Rlock.acquire() the resource is locked
    try:
        print("A: acquired lock1, waiting for lock2")
        time.sleep(0.1)
        lock2.acquire()
        try:
            print("A: acquired both locks")
        finally:
            lock2.release() # to release the lock, protecting resource management
    finally:
        lock1.release()
def thread_b():
    lock2.acquire()
    try:
        print("B: acquired lock2, waiting for lock1")
        time.sleep(0.1)
        lock1.acquire()
        try:
            print("B: acquired both locks")
        finally:
            lock1.release()
    finally:
        lock2.release()
t1 = threading.Thread(target=thread_a)
t2 = threading.Thread(target=thread_b)
t1.start()
t2.start()
t1.join(timeout=2)
t2.join(timeout=2)
print("Thread A alive:", t1.is_alive())
print("Thread B alive:", t2.is_alive())



A: acquired lock1, waiting for lock2
A: acquired both locks
B: acquired lock2, waiting for lock1
B: acquired both locks
Thread A alive: False
Thread B alive: False


| Feature | Lock | RLock |
| :-: | :-- | :-- |
| Reacquire Lock | Cannot be reacquired by the same thread | Can be acquired multiple times by the same thread |
| Releasing lock | Can be released by any thread | Can only be released by the acquiring thread |
| Ownership | Owner by one thread at a time | Owned by the acquiring thread, multiple times |
| Execution speed | Faster execution | Slower due to tracking acquisition count |

## 13. threading.Event
`threading.Event` is used to communicate between threads.

In [18]:
def task(event: threading.Event, thread_id: int) -> None:
    print(f"Thread {thread_id} started. Waiting for the signal...")
    event.wait()
    print(f"Received signal. Thread {thread_id} is finishing.")

def main() -> None:
    event = threading.Event()
    t1 = threading.Thread(target=task, args=(event, 1))
    t2 = threading.Thread(target=task, args=(event, 2))

    t1.start()
    t2.start()

    print("Blocking the main thread for 3 seconds...")
    time.sleep(3)
    event.set()

    t1.join()
    t2.join()
    print("Both threads have finished.")

main()


Thread 1 started. Waiting for the signal...
Thread 2 started. Waiting for the signal...
Blocking the main thread for 3 seconds...
Received signal. Thread 2 is finishing.
Received signal. Thread 1 is finishing.
Both threads have finished.


## 14. ThreadPoolExecutor


In [19]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def task(n):
    time.sleep(1)  # Simulate waiting for I/O
    return f"Task {n} finished"

# Without ThreadPoolExecutor: tasks run one at a time
start = time.time()
results = [task(n) for n in range(4)]
print("Without:", results, f"({time.time() - start:.1f}s)")

# With ThreadPoolExecutor: tasks can run concurrently
start = time.time()
with ThreadPoolExecutor(max_workers=4) as executor:
    results = list(executor.map(task, range(4)))
print("With:", results, f"({time.time() - start:.1f}s)")


Without: ['Task 0 finished', 'Task 1 finished', 'Task 2 finished', 'Task 3 finished'] (4.0s)
With: ['Task 0 finished', 'Task 1 finished', 'Task 2 finished', 'Task 3 finished'] (1.0s)


In [21]:
# Worker function
def compute_square(n):
    """Compute the square of a number with simulated delay."""
    time.sleep(0.5)  # Simulate work
    return n, n ** 2
    
def main():
    # Input numbers
    numbers = [1, 2, 3, 4, 5, 6, 7]

    # Create ThreadPoolExecutor with 3 worker threads
    with ThreadPoolExecutor(max_workers=3) as executor:
        # Submit tasks
        futures = [executor.submit(compute_square, num) for num in numbers]

        # Process results as they complete
        for future in as_completed(futures):
            num, result = future.result()
            print(f"Input: {num} -> Result: {result}")

main()


Input: 2 -> Result: 4
Input: 1 -> Result: 1
Input: 3 -> Result: 9
Input: 5 -> Result: 25
Input: 4 -> Result: 16
Input: 6 -> Result: 36
Input: 7 -> Result: 49


1. `ThreadPoolExecutor` - creates a pool of threads `(max_workers=3)` to run tasks concurrently
2. `executor.submit()` - schedules each task and returns a `Future` object.
3. `as_completed()` iterates over futures as they finish, so results are printed in completion order.

## 15. Local Caching
Store frequently-used data locally instead of recalculating.

In [22]:
cache = {}

def get_user(user_id):
    if user_id not in cache:
        cache[user_id] = f"user {user_id}"
    return cache[user_id]
# Example
print(get_user(42))  # user 42
print(get_user(42))  # user 42 (returned from cache)


user 42
user 42


Requests to servers add up but when data is stored locally, the requests are cheaper and faster.